# Credit Card Fraud Detection — Exploratory Data Analysis

**Portfolio-ready EDA based on my original Hackveda Task 2 notebook.**

This project explores transaction distributions, class imbalance, outliers, feature relationships, PCA-based visualization, Isolation Forest anomaly detection, and fraud activity over time.

> **Dataset requirement:** place `creditcard.csv` in `../data/` when running this notebook locally.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42


## 2. Load the Dataset

In [ ]:
DATA_PATH = Path("../data/creditcard.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place creditcard.csv in the project's data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Data Quality and Structure

In [ ]:
print("Data types and non-null counts:")
display(df.info())

print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0] if missing.any() else pd.Series({"Result": "No missing values"}))

print("\nSummary statistics:")
display(df.describe().T)


## 4. Target Variable and Class Imbalance

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index().mul(100)

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_pct.round(4)
}, index=["Non-Fraud (0)", "Fraud (1)"])

display(class_summary)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Class")
ax.set_title("Class Distribution")
ax.set_xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
ax.set_ylabel("Transactions")
plt.tight_layout()
plt.show()


## 5. Transaction Amount and Outliers

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df["Amount"])
plt.title("Transaction Amount — Boxplot")
plt.xlabel("Amount")
plt.tight_layout()
plt.show()

# Corrected Z-score logic: a row is flagged when ANY numeric feature exceeds |Z| > 3.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [c for c in numeric_cols if c != "Class"]

z = np.abs((df[feature_cols] - df[feature_cols].mean()) / df[feature_cols].std(ddof=0))
outlier_mask = z.gt(3).any(axis=1)

print(f"Rows with at least one |Z-score| > 3: {outlier_mask.sum():,}")
print(f"Share of rows flagged: {outlier_mask.mean()*100:.2f}%")


## 6. Feature Scaling

In [ ]:
X = df.drop(columns=["Class"]).copy()
y = df["Class"].copy()

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

display(X_scaled.head())


## 7. Feature Distributions

In [ ]:
# Plot a manageable selection first; the anonymized V-features are numerous.
selected_features = [c for c in ["Time", "Amount", "V1", "V2", "V3", "V4"] if c in X_scaled.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, selected_features):
    sns.histplot(X_scaled[col], bins=30, kde=False, ax=ax)
    ax.set_title(f"Distribution: {col}")
for ax in axes.flat[len(selected_features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 8. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.1)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

target_corr = corr["Class"].drop("Class").abs().sort_values(ascending=False).head(10)
display(target_corr.to_frame("Absolute Correlation with Class"))


## 9. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Class": y
})

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by 2 PCs:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=pca_df.sample(min(10000, len(pca_df)), random_state=RANDOM_STATE),
    x="PC1", y="PC2", hue="Class", alpha=0.45, s=25
)
plt.title("PCA Projection of Transactions")
plt.tight_layout()
plt.show()


## 10. Isolation Forest Anomaly Detection

In [ ]:
# The original notebook used contamination=0.01.
# We retain that analytical choice while making the result reproducible.
iso_forest = IsolationForest(
    contamination=0.01,
    random_state=RANDOM_STATE,
    n_estimators=200,
    n_jobs=-1
)

anomaly_pred = iso_forest.fit_predict(X_scaled)
analysis_df = df.copy()
analysis_df["Anomaly"] = anomaly_pred
analysis_df["Anomaly_Label"] = np.where(anomaly_pred == -1, "Anomaly", "Normal")

display(analysis_df["Anomaly_Label"].value_counts())

plt.figure(figsize=(7, 5))
sns.countplot(data=analysis_df, x="Anomaly_Label")
plt.title("Isolation Forest Anomaly Detection")
plt.tight_layout()
plt.show()


## 11. Fraud Activity Over Time

In [ ]:
# Time is recorded in seconds from the start of the observation period.
analysis_df["Hour"] = (analysis_df["Time"] // 3600).astype(int)

fraud_per_hour = (
    analysis_df.loc[analysis_df["Class"] == 1]
    .groupby("Hour")
    .size()
    .sort_index()
)

average_fraud_cases_per_hour = fraud_per_hour.mean()
peak_hour = fraud_per_hour.idxmax()
peak_fraud_cases = fraud_per_hour.max()

print(f"Average fraud cases per active hour: {average_fraud_cases_per_hour:.2f}")
print(f"Peak observed hour index: {peak_hour} ({peak_fraud_cases} fraud cases)")

plt.figure(figsize=(12, 5))
fraud_per_hour.plot(kind="bar")
plt.title("Fraud Cases by Hour Index")
plt.xlabel("Hour Index")
plt.ylabel("Fraud Cases")
plt.tight_layout()
plt.show()


## 12. Key Findings and Conclusion

In [ ]:
fraud_rate = analysis_df["Class"].mean() * 100

print(f"Fraud rate: {fraud_rate:.2f}%")
print(f"Rows flagged by |Z| > 3 in at least one numeric feature: {outlier_mask.sum():,}")
print(f"Two-component PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"Isolation Forest anomaly count: {(analysis_df['Anomaly'] == -1).sum():,}")


### Portfolio interpretation

- The target is **highly imbalanced**, so fraud detection should not be evaluated with accuracy alone.
- `Amount` shows substantial variation and is useful for exploratory outlier analysis.
- PCA provides a two-dimensional view of the high-dimensional transaction features.
- Isolation Forest provides an **unsupervised anomaly signal**; an anomaly is not automatically a confirmed fraud case.
- The next stage after EDA would be supervised modeling with imbalance-aware evaluation such as precision, recall, F1-score and PR-AUC.

### Important scope note

This project is an **EDA and anomaly-analysis project**, not a production fraud detection system.
